In [4]:
import pandas as pd
from data_profiling import ProfileReport

In [23]:
df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")
df.describe()

/tmp/ipykernel_52682/1620990980.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")


,siren_amenageur,nbre_pdc,puissance_nominale,consolidated_longitude,consolidated_latitude,consolidated_code_postal
count,1.362470e+05,227232.000000,227232.000000,227232.000000,227232.000000,134362.000000
mean,6.924200e+08,14.983180,102.212347,2.678738,46.733937,52603.766683
std,2.572975e+08,47.103571,579.594838,4.529209,4.034939,26780.472001
min,0.000000e+00,1.000000,0.000000,-149.905377,-44.996198,1000.000000
25%,5.243353e+08,2.000000,22.000000,0.774255,44.871519,31112.500000
50%,8.427185e+08,4.000000,22.000000,2.412432,47.340547,57160.000000
75%,8.951636e+08,9.000000,100.000000,4.836760,48.865021,76410.000000
max,9.921637e+08,505.000000,160000.000000,166.462000,61.520355,97418.000000


In [6]:
df.columns

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id',
       'datagouv_resource_id', 'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
     

In [7]:
profile = ProfileReport(df, title="Profiling Report")

In [9]:
report_path = "rapport_data_profiling.html"
profile.to_file(report_path)

print(f"Rapport généré : {report_path}")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 54.58it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 54.58it/s]

Rapport généré : rapport_data_profiling.html


In [11]:
def quality_score(df):
    total_cells = df.shape[0] * df.shape[1]

    missing = df.isna().sum().sum()
    completeness = 1 - missing / total_cells

    duplicates = df.duplicated().sum() / len(df)

    numeric_outliers = 0
    for col in df.select_dtypes("number"):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        numeric_outliers += outliers

    outlier_ratio = numeric_outliers / total_cells

    score = (
        0.5 * completeness +
        0.3 * (1 - duplicates) +
        0.2 * (1 - outlier_ratio)
    )

    return round(score * 100, 2)

print(quality_score(df))

93.77


93.77% des données semblent de qualité en se basant sur le nombre de doublons, de valeurs aberrantes et de valeurs manquantes.

In [31]:
df["num_pdl"].isna().sum()

np.int64(111260)

In [16]:
df.sample(10)

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
27234,DREAM ENERGY,980819866.0,support@dream-energy.fr,DREAM ENERGY,support@dream-energy.fr,tel:+33-1-30-71-20-20,Route de Bar le Duc,FRDREP521006315,FRDREP521006315,Route de Bar le Duc,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,4.955462,48.649427,52100.0,Saint-Dizier,False,True,False
84150,Ionity,838436145.0,info@ionity.eu,Ionity,info@ionity.eu,tel:+33-1-87-21-08-91,IONITY GmbH IONITY Maison Dieu,FRIOYP13530890,FRIOYP13530890,IONITY GmbH IONITY Maison Dieu,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,4.035971,47.504363,89420.0,Saint-André-en-Terre-Plaine,False,True,False
178582,2_AM0001_SMOYS,NaN,NaN,NaN,cpo@citeos.com,+33 1 41 44 73 53,CPO CITEOS SMOYS,FRS91PVDBWY,a03fa8bc-c45a-54ed-aaba-ab91a39a6f1a,Arpajon - Rue Édouard Robert,...,4d6551b9-bab7-46f8-9904-f46d565e68c6,citeos-ingenierie-idf-est-cogelum-idf,2025-04-30T09:55:33.008000+00:00,2.246670,48.590658,NaN,NaN,False,False,False
36851,SPBR1,882332562.0,contact@eborn.fr,EASYCHARGE,contact@eborn.fr,tel:+33-4-23-10-03-50,eborn,FREBNPBPYMA,0fe84613-0e12-5d3c-b9c5-8c2ce94eb084,AIX LES BAINS - Aquarium,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,5.889652,45.693716,73100.0,Aix-les-Bains,True,True,False
225386,Zephyre,NaN,NaN,Zephyre | FR*ZP1,sav@zephyre.fr,NaN,Zephyre,FRZP1P2630504714866983528,601128,Zephyre/LP009791,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,2.808255,48.855147,NaN,NaN,False,False,False
60230,Burger King - Essey-lès-Nancy,NaN,NaN,Freshmile | FR*FR1,roaming@freshmile.com,NaN,Freshmile France,FRFR1P7889116791604010231,466254,Freshmile France/VY4NRBJE2L,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,6.241860,48.710818,NaN,NaN,False,False,False
69585,Grand Lyon,419070180.0,lucas.malacarne@izivia.com,IZIVIA,sav@izivia.com,+33(0)972668001,Grand Lyon,FRGLYPLYON121,FRSODPLYON1211,LY806 - Berliet - Epargne,...,e297f18c-bb35-445f-af43-0217c27ab4fe,izivia,2023-11-23T13:01:13.994000+00:00,4.860316,45.742073,NaN,NaN,False,False,False
120297,EDF-EV100,NaN,NaN,Izivia,sav@izivia.com,0972668001,EDF-EV100,FROTHPIZIG598171,FR*SOD*S*IZIG*598*17*_*_,CNPE CATTENOM - CHADWICK 2-2,...,20e9f13e-a3d4-470a-a356-5714b91cddce,izivia,2021-12-30T10:45:19.638000+00:00,6.220030,49.412200,NaN,NaN,False,False,False
194104,TotalEnergies Charging Services,844192443.0,supervision-ev.france@totalenergies.com,TotalEnergies Charging Services,supervision-ev.france@totalenergies.com,01 41 35 40 00,TotalEnergies - Metpark,FRTCBPMETAME,FR*TCB*PMETAME,METPARK | TotalEnergies | Parking Amédée,...,17bdbce9-8a82-4322-a5e5-675c78a33281,totalenergies-marketing-france,2025-10-09T16:16:08.755000+00:00,-0.561459,44.819914,33800.0,Bordeaux,True,True,True
90261,IZIVIA FAST,951478437.0,lucas.malacarne@izivia.com,IZIVIA,sav@izivia.com,+33(0)972668001,IZIVIA FAST,FRIZFPFAST576,FRSODPFAST5761,IZIVIA FAST - McDonald's - BRUAY LA BUISSIERE,...,7696ac9e-2789-4eb6-9091-8a9b44cf0721,izivia,2023-11-30T09:39:27.928000+00:00,2.576102,50.495020,62700.0,Bruay-la-Buissière,True,True,False


In [ ]:
columnes_a_verifier = [
    col for col in df.columns
    if "id_pdc" in col.lower()
    or "itiner" in col.lower()
    or "station" in col.lower()
    or "date" in col.lower()
    or "operateur" in col.lower()
]

print("Colonnes candidates:")
print(columnes_a_verifier)

col_id = "id_pdc_itinerance"
print("\nNombre de lignes:", len(df))
print("Valeurs distinctes de id_pdc_itinerance:", df[col_id].nunique(dropna=False))
print("Lignes en doublon sur id_pdc_itinerance:", df.duplicated(subset=[col_id]).sum())

freq_id = df[col_id].value_counts(dropna=False)
print("\nTop 10 des valeurs les plus fréquentes:")
print(freq_id.head(10))

if len(columnes_a_verifier) > 1:
    print("\nTest de granularité sur plusieurs colonnes:")
    for taille in range(2, min(5, len(columnes_a_verifier)) + 1):
        subset = columnes_a_verifier[:taille]
        doublons = df.duplicated(subset=subset).sum()
        print(f"{subset} -> doublons: {doublons}")

Colonnes candidates:
['nom_operateur', 'contact_operateur', 'telephone_operateur', 'id_station_itinerance', 'id_station_local', 'nom_station', 'implantation_station', 'adresse_station', 'id_pdc_itinerance', 'id_pdc_local', 'station_deux_roues', 'date_mise_en_service', 'date_maj', 'consolidated_longitude', 'consolidated_latitude', 'consolidated_code_postal', 'consolidated_commune', 'consolidated_is_lon_lat_correct', 'consolidated_is_code_insee_verified', 'consolidated_is_code_insee_modified']

Nombre de lignes: 227232
Valeurs distinctes de id_pdc_itinerance: 160138
Lignes en doublon sur id_pdc_itinerance: 67094

Top 10 des valeurs les plus fréquentes:
id_pdc_itinerance
Non concerné       118
FRLIBE3126085        4
FRALLEGO8005071      3
FRALLEGO8005072      3
FRUBIE10021422       3
FRALLEGO8005251      3
FRALLEGO8005252      3
FRUBIE10021897       3
FRALLEGO8005271      3
FRALLEGO8005272      3
Name: count, dtype: int64

Test de granularité sur plusieurs colonnes:
['nom_operateur', 'con

In [28]:
id_exemple = freq_id.index[1]
exemple = df.loc[df[col_id] == id_exemple, [
    col for col in [
        "id_station_itinerance",
        "id_pdc_itinerance",
        "id_pdc_local",
        "nom_station",
        "nom_operateur",
        "date_mise_en_service",
        "date_maj",
        "consolidated_commune",
        "consolidated_code_postal",
    ] if col in df.columns
]]
print(f"Exemple pour {id_exemple}:")
display(exemple)

Exemple pour FRLIBE3126085:


,id_station_itinerance,id_pdc_itinerance,id_pdc_local,nom_station,nom_operateur,date_mise_en_service,date_maj,consolidated_commune,consolidated_code_postal
105425,FRLIBE3126085,FRLIBE3126085,frlibe3126085,Résidence Carouge,Bornevo,2023-06-23,2023-10-21,Brétigny-sur-Orge,91220.0
105426,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,NaN,2023-11-10,2023-11-10,NaN,NaN
105427,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0
105428,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0


id_pdc_itinerance n'est pas une clef car une ligne représente un snapshot de l'évolution de la mission d'un opérateur. Donc une itinérance peut apparaitre pour plusieurs dates différentes, nom d'opérateur, point de charge local.